# Trotter Ordering Errors For H, H*, And H2

This notebook compares three methods from `analysis/ordering.py`:

- `None`: keep the original QHAT/OpenFermion term order
- `lexicographical`: sort dense Pauli strings
- `group_evolve_greedy`: color the noncommutation graph and make each color contiguous

Atomic H uses the 6-31G basis so there are two spatial orbitals. H is the lowest distinct energy in the one-electron sector; H* is the next distinct orbital energy, skipping spin degeneracy. H2 uses STO-3G at 0.7414 Angstrom and the ground state in the two-electron sector.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next(path for path in [cwd, *cwd.parents] if (path / 'analysis' / 'ordering.py').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.examples.group_coloring_ordering_demo import (
    build_greedy_coloring_groups,
    group_is_pairwise_commuting,
)
from analysis.examples.ordering_error_comparison import (
    ORDERING_METHODS,
    build_evolution_cases,
    print_results,
    run_ordering_experiments,
)
from analysis.ordering import reorder_paulis

## 1. Build Physical States In Fixed-Particle Sectors

The Hamiltonian is diagonalized only in the sector with the physical electron count. This avoids accidentally selecting a zero-, three-, or four-electron eigenstate from the full qubit Hilbert space.

In [ ]:
cases = build_evolution_cases()

for case in cases:
    print(f'{case.name:10s} E = {case.energy:+.12f} Hartree, particles = {case.particle_number}')
    print(f'           {case.basis}; {len(case.terms)} Pauli terms')

## 2. Inspect The Greedy Commuting Groups

QHAT's graph connects **noncommuting** terms. Adjacent graph nodes cannot share a color, so each color class is pairwise commuting.

In [ ]:
for case in (cases[0], cases[2]):
    graph, coloring, groups = build_greedy_coloring_groups(case.terms)
    print(f'\n{case.name}: nodes={graph.number_of_nodes()}, edges={graph.number_of_edges()}, colors={len(groups)}')
    for color, group in enumerate(groups):
        labels = ', '.join(pauli for pauli, _ in group)
        print(f'  color {color}: commuting={group_is_pairwise_commuting(group)}: {labels}')

## 3. Compare Exact And Trotter Evolution

We use one first-order step at `t=1` Hartree$^{-1}$. The coarse step is intentional: it makes ordering effects visible.

- **operator error**: spectral norm $\|U_T-U\|_2$, independent of the chosen state
- **state error**: $\|(U_T-U)|\psi\rangle\|_2$
- **infidelity**: $1-|\langle\psi_T|\psi_{exact}\rangle|^2$, insensitive to global phase

In [ ]:
time = 1.0
num_steps = 1
method = 1

results = run_ordering_experiments(
    cases=cases,
    ordering_methods=ORDERING_METHODS,
    time=time,
    num_steps=num_steps,
    method=method,
)
print_results(results, time, num_steps, method)

## 4. See The Actual Pauli Orders

H and H* use the same Hamiltonian, so they have the same operator error and Pauli order. Their state-dependent errors can differ because they begin in different eigenstates.

In [ ]:
for case in (cases[0], cases[2]):
    print(f'\n{case.name}')
    for ordering in ORDERING_METHODS:
        label = 'none' if ordering is None else ordering
        ordered = reorder_paulis(case.terms, ordering)
        print(f'  {label:20s}: ' + ' -> '.join(ordered))

## 5. Experiment Further

Increase `num_steps` to verify convergence, or set `method = 2` for QHAT's symmetric second-order formula. Graph coloring guarantees commuting color classes; it does **not** guarantee the globally smallest Trotter error. The numerical table is what tells us whether a particular grouped order helps for a particular Hamiltonian and state.